# Lecture 07: Compute budgets and scaling

Four **ungraded** exercises for October 21. All students use the same practice.
The examples run offline on CPU with Python's standard library. No model training,
API key, GPU allocation, or additional assignment is required.

Every loss value here is **synthetic**, generated by the formula in
`assets/README.md`. These are not Jiaxuan Zou's results or paper measurements.
Predict first, then run the supplied calculation.

The first two periods prepare for Jiaxuan's guest session on
[scientific scaling ladders](https://jiaxuanzou0714.github.io/blog/2026/how-to-build-scientific-scaling-ladder/).
The guest session is 15:25–16:10 (Asia/Shanghai).

In [ ]:
from pathlib import Path
import csv
import math
import sys
from statistics import mean

assets = Path("assets")
assert assets.is_dir(), "Open this notebook with its assets folder beside it."
sys.path.insert(0, str(assets.resolve()))
from scaling import (FLOOR, REFERENCE_COMPUTE, training_flops,
                     persistent_bytes, toy_loss, optimal_allocation,
                     fit_power_law, predict_loss)
print("Standard-library examples ready. All loss data are synthetic.")

## E01 · Which run costs more? · 4 minutes

Run A: 50 million parameters, 4 billion processed tokens.
Run B: 200 million parameters, 1 billion processed tokens.

1. Estimate each with `C = 6ND`.
2. Count persistent FP32 weights, gradients, and two Adam moments.
3. Estimate compute time for an **assumed** sustained 100 TFLOP/s.

Why might memory, elapsed time, and loss differ despite equal FLOPs?
The approximation omits attention-specific costs, activation memory,
recomputation, and temporary buffers.

### Why the factor six?

![Forward and two backward matrix products, with dimensions and approximate costs.](assets/training-matmul.svg)

Here `G = ∂L/∂Y` is the incoming gradient. Each product costs about
`2mdh` FLOPs for `m` token positions and a layer with `dh` weights.
Adding the three costs and summing over dense layers and all training
tokens gives the `6ND` approximation.


In [ ]:
runs = [("A", 50e6, 4e9), ("B", 200e6, 1e9)]
for name, parameters, tokens in runs:
    compute = training_flops(parameters, tokens)
    gb = persistent_bytes(parameters) / 1e9
    hours = compute / 100e12 / 3600
    print(f"{name}: {compute:.2e} FLOPs; {gb:.1f} GB; {hours:.2f} hours")
assert training_flops(*runs[0][1:]) == training_flops(*runs[1][1:])

<details><summary>Discussion after calculating</summary>

Both use 1.2e18 approximate FLOPs. Persistent arrays require 0.8 GB and 3.2 GB
(decimal units), excluding activations and other memory. The illustrative time
is 3.33 hours only if both sustain 100 TFLOP/s. Actual utilization, overhead,
and statistical performance can differ.

</details>

## E02 · Four times the budget · 4 minutes

Use the invented surface:

`L = 1.5 + 0.8/sqrt(N/1e8) + 0.8/sqrt(D/2e9)`.

At `C0 = 1.2e18`, the optimum is 100M parameters and 2B tokens.
Predict the allocation at 4C0. Compare with increasing only N fourfold.
These coefficients have no empirical status.

In [ ]:
for multiple in [1, 4, 16]:
    n, d = optimal_allocation(multiple)
    assert math.isclose(training_flops(n, d), multiple * REFERENCE_COMPUTE)
    print(f"{multiple}x: N={n/1e6:.0f}M, D={d/1e9:.0f}B, loss={toy_loss(n,d):.6f}")

balanced = toy_loss(200e6, 4e9)
parameters_only = toy_loss(400e6, 2e9)
print("At 4x:", balanced, "versus", parameters_only)
assert balanced < parameters_only

# Independently search a grid at the same compute budget.
grid = [10 ** (7 + i / 500) for i in range(1501)]
budget = 4 * REFERENCE_COMPUTE
best = min(grid, key=lambda n: toy_loss(n, budget / (6 * n)))
print(f"Grid minimum: {best / 1e6:.2f}M parameters")
assert abs(math.log(best / 200e6)) < math.log(10) / 500

<details><summary>Discussion after calculating</summary>

Equal exponents in this surface give N and D proportional to sqrt(C).
The 4x optimum is 200M/4B, loss 2.631371. Spending everything on N gives
400M/2B and loss 2.700000. Changing coefficients, exponents, data regime,
or cost objective can change the recommendation.

</details>

## E03 · Predict a larger run · 7 minutes

Fit `L = E + K*x**(-gamma)`, where x=C/C0 and the supplied floor E=1.5.
Use only **fit** rows at x=0.25,1,4,16. Average the three invented replicates
at each budget before taking the logarithm. Fit a line to
`log(mean L - E)` versus `log(x)`.

Before revealing the holdout at x=64, write a prediction. Predeclare an
absolute tolerance of **0.05 nats/token** for this exercise. This classroom
rule is not a confidence interval or a research recommendation. In real
experiments the floor is usually unknown.

![Fit on small runs, freeze the recipe and curve and tolerance, write the prediction, then check the reserved run.](assets/prediction-check.svg)


In [ ]:
with (assets / "synthetic-runs.csv").open() as handle:
    rows = list(csv.DictReader(handle))
assert all(row["kind"] == "synthetic" for row in rows)
fit_rows = [row for row in rows if row["split"] == "fit"]
levels = sorted({float(row["compute_multiple"]) for row in fit_rows})
fit_points = [(x, mean(float(row["loss_nats"]) for row in fit_rows
                       if float(row["compute_multiple"]) == x)) for x in levels]
coefficient, exponent = fit_power_law(fit_points, floor=FLOOR)
prediction = predict_loss(64, coefficient, exponent, floor=FLOOR)
tolerance = 0.05
print(f"K={coefficient:.6f}; gamma={exponent:.6f}; prediction={prediction:.6f}")
assert 64 not in levels

### Reveal the holdout after writing the prediction

Do not add these rows to the fit and call the result independent validation.
If they guide a revised recipe or fit, they become development data.

In [ ]:
holdout = [row for row in rows if row["split"] == "holdout"]
assert {float(row["compute_multiple"]) for row in holdout} == {64.0}
actual = mean(float(row["loss_nats"]) for row in holdout)
error = abs(actual - prediction)
print(f"Synthetic holdout mean: {actual:.6f}")
print(f"Absolute error: {error:.6f}; tolerance: {tolerance:.2f}")
print("Decision:", "passes this check" if error <= tolerance else "diagnose before scaling")
assert math.isclose(error, 0.16, abs_tol=1e-10)

<details><summary>Interpretation and construction</summary>

The fit recovers K=1.6, gamma=0.25. Prediction: 2.065685.
Synthetic holdout mean: 2.225685. Error: 0.16, exceeding the 0.05 rule.
The four fitting means lie exactly on the curve, illustrating why fit quality
alone does not validate extrapolation.

The generator adds offsets -0.01,0,+0.01 at each budget and an extra +0.16
at x=64. This deliberate shift has no diagnosed physical cause. Real checks
could examine stability, changed data, accounting, or an intermediate scale.
New independent evidence is needed after revising the model.

</details>

## E04 · A small experiment proposal · 3 minutes

Allocate 12 runs between two recipes, smaller fitting scales, repeats, and a
larger validation condition. State the fixed evaluation protocol and a reason
to delay a large run. Price the runs as well as counting them.
This is a planning exercise, not a request to launch jobs.

In [ ]:
plan = [(recipe, scale, seed, split)
        for recipe in ["baseline", "candidate"]
        for scale, split in [(0.25, "fit"), (1, "fit"), (4, "holdout")]
        for seed in [1, 2]]
assert len(plan) == 12
fit_count = sum(split == "fit" for _, _, _, split in plan)
holdout_count = sum(split == "holdout" for _, _, _, split in plan)
total_compute = sum(scale * REFERENCE_COMPUTE for _, scale, _, _ in plan)
print(f"{fit_count} fitting runs, {holdout_count} validation runs")
print(f"Planning compute: {total_compute:.2e} FLOPs ({total_compute/REFERENCE_COMPUTE:.0f} x C0)")

<details><summary>One acceptable response</summary>

Two recipes × two fitting scales × two seeds = eight fitting runs.
Two recipes × one larger scale × two seeds = four validation runs.
The listed budgets total **21 C0 = 2.52e19 approximate FLOPs**.
A twelve-run limit alone does not guarantee affordability.

Freeze data, tokenizer, evaluation corpus, aggregation, and declared scaling
rules. Change one candidate factor. Set a decision tolerance before validation.
Delay the larger run if the ranking changes, uncertainty is too large, training
becomes unstable, or the forecast misses the tolerance.

Two fitting scales and two seeds provide limited research evidence and cannot
identify a general five-parameter loss surface. Preserve a smaller fallback.

</details>

## Reading and guest discussion

- [Hoffmann et al., §3](https://arxiv.org/html/2203.15556v1): allocation and IsoFLOP experiments.
- [Porian et al., §§3–4](https://arxiv.org/html/2406.19146v4): accounting and optimization confounds.
- [Jiaxuan Zou's article](https://jiaxuanzou0714.github.io/blog/2026/how-to-build-scientific-scaling-ladder/): prepare a question about ladder design or validation.

Our ordinary least-squares fit in log excess loss with a known floor does not
reproduce Chinchilla's robust nonlinear estimator.